# IMDb Official Dataset Enrichment

This notebook enriches the streaming catalog using IMDb's official non-commercial datasets.

## Enrichment scope

- **Movies**
  - Principal cast
  - Directors
  - Cinematographers / Directors of Photography
- **TV series**
  - Principal cast only

## Matching strategy

1. Use an existing valid `imdb_id` when available.
2. Validate that ID against `title.basics.tsv.gz`.
3. For rows without a valid IMDb ID, match against IMDb using:
   - `title`
   - `original_title`
   - `release_year`
   - `content_type`
4. Keep ambiguous matches unresolved rather than guessing.

## Files

Input:

`DATA/PROCESSED/all_streaming_titles.csv`

Output:

`DATA/PROCESSED/all_streaming_titles_enriched.csv`

IMDb requires the following acknowledgement for permitted non-commercial dataset use:

> Information courtesy of IMDb. Used with permission.

Review IMDb's current dataset terms before using or redistributing the result.


## 1. Install dependencies

In [22]:
%pip install -q pandas requests tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [23]:
from __future__ import annotations

import json
import re
import time
import unicodedata

from pathlib import Path
from typing import Any

import pandas as pd
import requests

from tqdm.auto import tqdm


## 3. Configuration

The IMDb files are large. This notebook reads them in chunks and only keeps records relevant to your catalog.


In [24]:
# Project paths
INPUT_PATH = Path("DATA/PROCESSED/all_streaming_titles.csv")
OUTPUT_PATH = Path("DATA/PROCESSED/all_streaming_titles_enriched.csv")
IMDB_DATA_DIR = Path("DATA/RAW/IMDB")

# Intermediate files
TITLE_MATCHES_PATH = Path("DATA/INTERIM/imdb_title_matches.csv")
RELEVANT_PRINCIPALS_PATH = Path("DATA/INTERIM/imdb_relevant_principals.csv")
RELEVANT_CREW_PATH = Path("DATA/INTERIM/imdb_relevant_crew.csv")
RELEVANT_NAMES_PATH = Path("DATA/INTERIM/imdb_relevant_names.csv")

# Official IMDb non-commercial dataset files
IMDB_BASE_URL = "https://datasets.imdbws.com"

DATASET_FILES = {
    "title_basics": "title.basics.tsv.gz",
    "title_principals": "title.principals.tsv.gz",
    "title_crew": "title.crew.tsv.gz",
    "name_basics": "name.basics.tsv.gz",
}

# Reading configuration
CHUNK_SIZE = 500_000

# Maximum principal cast members saved per title
MAX_CAST = 10

# Test mode:
# Use an integer such as 100 for a quick pipeline test.
# Use None for the complete catalog.
ROW_LIMIT = None

# Strict matching controls
ALLOW_YEAR_DIFFERENCE = 0
ALLOW_UNIQUE_YEAR_PLUS_MINUS_ONE_FALLBACK = True

# Existing IMDb IDs must follow this format
IMDB_ID_PATTERN = re.compile(r"^tt\d{7,10}$")

# IMDb title types corresponding to the catalog content types
MOVIE_TITLE_TYPES = {
    "movie",
    "tvMovie",
}

TV_TITLE_TYPES = {
    "tvSeries",
    "tvMiniSeries",
}

# Only these principal categories are treated as cast
CAST_CATEGORIES = {
    "actor",
    "actress",
    "self",
}

# Cinematography detection
CINEMATOGRAPHY_CATEGORIES = {
    "cinematographer",
}

CINEMATOGRAPHY_JOB_PATTERNS = (
    "cinematographer",
    "director of photography",
)

IMDB_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TITLE_MATCHES_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)
print("IMDb directory:", IMDB_DATA_DIR)


Input: DATA/PROCESSED/all_streaming_titles.csv
Output: DATA/PROCESSED/all_streaming_titles_enriched.csv
IMDb directory: DATA/RAW/IMDB


## 4. Download official IMDb dataset files

In [25]:
def download_file(
    url: str,
    destination: Path,
    *,
    chunk_bytes: int = 1024 * 1024,
) -> None:
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already downloaded: {destination.name}")
        return

    print(f"Downloading {destination.name}...")

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get("Content-Length", 0))

        with destination.open("wb") as file, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_bytes):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))


for dataset_name, filename in DATASET_FILES.items():
    download_file(
        f"{IMDB_BASE_URL}/{filename}",
        IMDB_DATA_DIR / filename,
    )


Already downloaded: title.basics.tsv.gz
Already downloaded: title.principals.tsv.gz
Already downloaded: title.crew.tsv.gz
Already downloaded: name.basics.tsv.gz


## 5. Load and validate the streaming catalog

In [26]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_PATH.resolve()}"
    )

catalog = pd.read_csv(INPUT_PATH, low_memory=False)

required_columns = {
    "title",
    "original_title",
    "release_year",
    "content_type",
    "imdb_id",
}

missing_columns = required_columns.difference(catalog.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

catalog["content_type"] = (
    catalog["content_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

catalog["release_year"] = pd.to_numeric(
    catalog["release_year"],
    errors="coerce",
).astype("Int64")

if ROW_LIMIT is not None:
    working_catalog = catalog.head(ROW_LIMIT).copy()
else:
    working_catalog = catalog.copy()

print(f"Complete catalog rows: {len(catalog):,}")
print(f"Rows selected for this run: {len(working_catalog):,}")
print(working_catalog["content_type"].value_counts(dropna=False))
working_catalog.head(3)


Complete catalog rows: 96,200
Rows selected for this run: 96,200
content_type
tv       88647
movie     7553
Name: count, dtype: Int64


,genre_names,origin_country,engagement_score,original_language,popularity_norm,genre_ids,title_age,tmdb_id,freshness_score,original_title,...,adult,tvdb_id,business_value_score,content_type,status,is_recent,network,production_companies,runtime_final,genre_clean
0,"Crime, Drama, Comedy",US,7.789985,English,0.916136,"[80, 18, 35]",8.0,79744.0,0.922481,The Rookie,...,0.0,NaN,65.265256,tv,Returning Series,0,NaN,"ABC Studios, Entertainment One, Lionsgate Tele...",NaN,"['Crime', 'Drama', 'Comedy']"
1,"Action, Science Fiction, Thriller",AU,2.340317,English,1.000000,"[28, 878, 53]",0.0,1265609.0,0.984496,War Machine,...,0.0,NaN,65.023056,movie,Released,1,NaN,"Lionsgate, Hidden Pictures, Huge Film, Range M...",110.0,"['Action', 'Science Fiction', 'Thriller']"
2,Drama,US,27.603475,English,0.796415,[18],21.0,1416.0,0.821705,Grey's Anatomy,...,0.0,NaN,64.772711,tv,Returning Series,0,NaN,"The Mark Gordon Company, shondaland, ABC Studi...",NaN,['Drama']


## 6. Normalize titles and IMDb IDs

In [27]:
LEADING_ARTICLES = {
    "a", "an", "the",
    "el", "la", "los", "las",
    "le", "les",
}


def normalize_title(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    text = str(value)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = text.lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def remove_leading_article(value: str) -> str:
    tokens = normalize_title(value).split()

    if tokens and tokens[0] in LEADING_ARTICLES:
        tokens = tokens[1:]

    return " ".join(tokens)


def normalize_imdb_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    text = str(value).strip()

    if text.endswith(".0") and text[:-2].isdigit():
        text = text[:-2]

    if text.isdigit():
        text = f"tt{text.zfill(7)}"

    return text if IMDB_ID_PATTERN.fullmatch(text) else ""


def expected_imdb_types(content_type: str) -> set[str]:
    if content_type == "movie":
        return MOVIE_TITLE_TYPES

    if content_type == "tv":
        return TV_TITLE_TYPES

    return set()


working_catalog["_catalog_row"] = working_catalog.index
working_catalog["_title_norm"] = (
    working_catalog["title"].apply(normalize_title)
)
working_catalog["_original_title_norm"] = (
    working_catalog["original_title"].apply(normalize_title)
)
working_catalog["_title_no_article"] = (
    working_catalog["title"].apply(remove_leading_article)
)
working_catalog["_original_no_article"] = (
    working_catalog["original_title"].apply(remove_leading_article)
)
working_catalog["_existing_imdb_id"] = (
    working_catalog["imdb_id"].apply(normalize_imdb_id)
)

print(
    "Rows with a valid-format existing IMDb ID:",
    working_catalog["_existing_imdb_id"].ne("").sum(),
)


Rows with a valid-format existing IMDb ID: 50590


## 7. Scan `title.basics` for existing IDs and missing-ID candidates

This step does two things in one pass:

- Confirms existing IMDb IDs.
- Keeps only IMDb titles whose normalized names could match rows missing an ID.


In [28]:
basics_path = IMDB_DATA_DIR / DATASET_FILES["title_basics"]

existing_ids = set(
    working_catalog.loc[
        working_catalog["_existing_imdb_id"].ne(""),
        "_existing_imdb_id",
    ]
)

rows_missing_id = working_catalog[
    working_catalog["_existing_imdb_id"].eq("")
].copy()

target_normalized_titles = set(
    rows_missing_id["_title_norm"]
).union(
    rows_missing_id["_original_title_norm"]
).union(
    rows_missing_id["_title_no_article"]
).union(
    rows_missing_id["_original_no_article"]
)

target_normalized_titles.discard("")

confirmed_existing_chunks = []
candidate_chunks = []

basics_columns = [
    "tconst",
    "titleType",
    "primaryTitle",
    "originalTitle",
    "startYear",
    "endYear",
    "isAdult",
]

reader = pd.read_csv(
    basics_path,
    sep="\t",
    dtype="string",
    na_values="\\N",
    usecols=basics_columns,
    chunksize=CHUNK_SIZE,
    compression="gzip",
)

for chunk in tqdm(reader, desc="Scanning title.basics"):
    relevant_types = chunk["titleType"].isin(
        MOVIE_TITLE_TYPES.union(TV_TITLE_TYPES)
    )
    chunk = chunk.loc[relevant_types].copy()

    existing_mask = chunk["tconst"].isin(existing_ids)

    if existing_mask.any():
        confirmed_existing_chunks.append(
            chunk.loc[existing_mask].copy()
        )

    if rows_missing_id.empty:
        continue

    chunk["_primary_norm"] = (
        chunk["primaryTitle"].apply(normalize_title)
    )
    chunk["_original_norm"] = (
        chunk["originalTitle"].apply(normalize_title)
    )
    chunk["_primary_no_article"] = (
        chunk["primaryTitle"].apply(remove_leading_article)
    )
    chunk["_original_no_article"] = (
        chunk["originalTitle"].apply(remove_leading_article)
    )

    candidate_mask = (
        chunk["_primary_norm"].isin(target_normalized_titles)
        | chunk["_original_norm"].isin(target_normalized_titles)
        | chunk["_primary_no_article"].isin(target_normalized_titles)
        | chunk["_original_no_article"].isin(target_normalized_titles)
    )

    if candidate_mask.any():
        candidate_chunks.append(
            chunk.loc[candidate_mask].copy()
        )

confirmed_existing = (
    pd.concat(confirmed_existing_chunks, ignore_index=True)
    if confirmed_existing_chunks
    else pd.DataFrame(columns=basics_columns)
)

title_candidates = (
    pd.concat(candidate_chunks, ignore_index=True)
    if candidate_chunks
    else pd.DataFrame(columns=basics_columns)
)

title_candidates["startYear"] = pd.to_numeric(
    title_candidates["startYear"],
    errors="coerce",
).astype("Int64")

print(f"Confirmed existing IDs: {len(confirmed_existing):,}")
print(f"Candidate IMDb titles for missing IDs: {len(title_candidates):,}")


Scanning title.basics: 26it [01:43,  3.98s/it]

Confirmed existing IDs: 49,867
Candidate IMDb titles for missing IDs: 70,637


## 8. Resolve IMDb IDs

The matching is deliberately strict:

- Correct movie/TV type.
- Exact normalized primary or original title.
- Same year whenever the catalog has a year.
- A ±1-year fallback is accepted only when the candidate is unique.


In [29]:
confirmed_existing_ids = set(
    confirmed_existing["tconst"].dropna()
)

candidate_indexes: dict[str, dict[str, list[int]]] = {
    "_primary_norm": {},
    "_original_norm": {},
    "_primary_no_article": {},
    "_original_no_article": {},
}

if not title_candidates.empty:
    for column in candidate_indexes:
        for candidate_index, value in title_candidates[column].items():
            if value:
                candidate_indexes[column].setdefault(
                    value,
                    [],
                ).append(candidate_index)


def collect_candidate_indices(row: pd.Series) -> set[int]:
    indexes = set()

    lookup_pairs = [
        ("_primary_norm", row["_title_norm"]),
        ("_original_norm", row["_title_norm"]),
        ("_primary_norm", row["_original_title_norm"]),
        ("_original_norm", row["_original_title_norm"]),
        ("_primary_no_article", row["_title_no_article"]),
        ("_original_no_article", row["_title_no_article"]),
        ("_primary_no_article", row["_original_no_article"]),
        ("_original_no_article", row["_original_no_article"]),
    ]

    for candidate_column, lookup_value in lookup_pairs:
        if lookup_value:
            indexes.update(
                candidate_indexes[candidate_column].get(
                    lookup_value,
                    [],
                )
            )

    return indexes


def resolve_missing_imdb_id(
    row: pd.Series,
) -> dict[str, Any]:
    expected_types = expected_imdb_types(
        str(row["content_type"])
    )

    if not expected_types:
        return {
            "imdb_resolved_id": "",
            "imdb_match_method": "",
            "imdb_match_status": "unsupported_content_type",
            "imdb_match_candidates": 0,
        }

    candidate_indices = collect_candidate_indices(row)

    if not candidate_indices:
        return {
            "imdb_resolved_id": "",
            "imdb_match_method": "",
            "imdb_match_status": "not_found",
            "imdb_match_candidates": 0,
        }

    candidates = title_candidates.loc[
        sorted(candidate_indices)
    ].copy()

    candidates = candidates[
        candidates["titleType"].isin(expected_types)
    ].copy()

    if candidates.empty:
        return {
            "imdb_resolved_id": "",
            "imdb_match_method": "",
            "imdb_match_status": "not_found_correct_type",
            "imdb_match_candidates": 0,
        }

    row_year = row["release_year"]

    if pd.notna(row_year):
        exact_year = candidates[
            candidates["startYear"].eq(int(row_year))
        ]

        if len(exact_year) == 1:
            match = exact_year.iloc[0]

            return {
                "imdb_resolved_id": match["tconst"],
                "imdb_match_method": "exact_title_type_year",
                "imdb_match_status": "matched",
                "imdb_match_candidates": 1,
            }

        if len(exact_year) > 1:
            return {
                "imdb_resolved_id": "",
                "imdb_match_method": "",
                "imdb_match_status": "ambiguous_exact_year",
                "imdb_match_candidates": len(exact_year),
            }

        if ALLOW_UNIQUE_YEAR_PLUS_MINUS_ONE_FALLBACK:
            nearby_year = candidates[
                candidates["startYear"].notna()
                & (
                    candidates["startYear"].sub(int(row_year))
                    .abs()
                    .le(1)
                )
            ]

            if len(nearby_year) == 1:
                match = nearby_year.iloc[0]

                return {
                    "imdb_resolved_id": match["tconst"],
                    "imdb_match_method": "exact_title_type_year_plus_minus_one",
                    "imdb_match_status": "matched",
                    "imdb_match_candidates": 1,
                }

            if len(nearby_year) > 1:
                return {
                    "imdb_resolved_id": "",
                    "imdb_match_method": "",
                    "imdb_match_status": "ambiguous_nearby_year",
                    "imdb_match_candidates": len(nearby_year),
                }

        return {
            "imdb_resolved_id": "",
            "imdb_match_method": "",
            "imdb_match_status": "year_mismatch",
            "imdb_match_candidates": len(candidates),
        }

    if len(candidates) == 1:
        match = candidates.iloc[0]

        return {
            "imdb_resolved_id": match["tconst"],
            "imdb_match_method": "unique_exact_title_type_no_year",
            "imdb_match_status": "matched",
            "imdb_match_candidates": 1,
        }

    return {
        "imdb_resolved_id": "",
        "imdb_match_method": "",
        "imdb_match_status": "ambiguous_no_year",
        "imdb_match_candidates": len(candidates),
    }


match_records = []

for _, row in tqdm(
    working_catalog.iterrows(),
    total=len(working_catalog),
    desc="Resolving IMDb IDs",
):
    existing_id = row["_existing_imdb_id"]

    if existing_id:
        if existing_id in confirmed_existing_ids:
            result = {
                "imdb_resolved_id": existing_id,
                "imdb_match_method": "existing_imdb_id",
                "imdb_match_status": "matched",
                "imdb_match_candidates": 1,
            }
        else:
            result = {
                "imdb_resolved_id": "",
                "imdb_match_method": "existing_imdb_id",
                "imdb_match_status": "existing_id_not_in_title_basics",
                "imdb_match_candidates": 0,
            }
    else:
        result = resolve_missing_imdb_id(row)

    match_records.append({
        "_catalog_row": row["_catalog_row"],
        **result,
    })

title_matches = pd.DataFrame(match_records)
title_matches.to_csv(TITLE_MATCHES_PATH, index=False)

working_catalog = working_catalog.merge(
    title_matches,
    on="_catalog_row",
    how="left",
    validate="one_to_one",
)

print(
    working_catalog["imdb_match_status"]
    .value_counts(dropna=False)
)


Resolving IMDb IDs: 100%|██████████| 96200/96200 [01:07<00:00, 1418.02it/s]


imdb_match_status
matched                            72489
not_found                          19347
year_mismatch                       2021
not_found_correct_type               881
existing_id_not_in_title_basics      707
ambiguous_exact_year                 436
ambiguous_no_year                    260
ambiguous_nearby_year                 59
Name: count, dtype: int64


## 9. Inspect title matching quality

In [30]:
matching_summary = (
    working_catalog
    .groupby(
        ["content_type", "imdb_match_status"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(
        ["content_type", "rows"],
        ascending=[True, False],
    )
)

matching_summary


,content_type,imdb_match_status,rows
4,movie,matched,7061
3,movie,existing_id_not_in_title_basics,355
5,movie,not_found,96
7,movie,year_mismatch,31
0,movie,ambiguous_exact_year,5
6,movie,not_found_correct_type,3
1,movie,ambiguous_nearby_year,1
2,movie,ambiguous_no_year,1
12,tv,matched,65428
13,tv,not_found,19251


In [31]:
working_catalog.loc[
    ~working_catalog["imdb_match_status"].eq("matched"),
    [
        "title",
        "original_title",
        "release_year",
        "content_type",
        "imdb_id",
        "imdb_match_status",
        "imdb_match_candidates",
    ],
].head(100)


,title,original_title,release_year,content_type,imdb_id,imdb_match_status,imdb_match_candidates
42,Obake no Q-tarō,オバケのQ太郎,1985,tv,NaN,year_mismatch,5
45,Pili,霹靂布袋戲,1988,tv,NaN,not_found_correct_type,0
53,Men on a Mission,아는 형님,2015,tv,NaN,not_found,0
111,Heartland,Heartland,2007,tv,NaN,ambiguous_exact_year,2
230,Raw,Raw,1993,tv,NaN,year_mismatch,3
...,...,...,...,...,...,...,...
1144,KinnPorsche: The Series,รักโคตรร้าย สุดท้ายโคตรรัก,2022,tv,NaN,not_found,0
1166,Ming Dynasty in 1566,大明王朝1566,2007,tv,NaN,not_found,0
1202,Cosmos,Cosmos,2014,tv,NaN,year_mismatch,1
1206,The Sindjelic Family,Синђелићи,2013,tv,NaN,not_found,0


## 10. Extract relevant principal credits

Only rows for resolved IMDb IDs are retained.

Cast categories:

- `actor`
- `actress`
- `self`

Cinematography is identified through an exact category or a specific job description.


In [32]:
resolved_ids = set(
    working_catalog.loc[
        working_catalog["imdb_match_status"].eq("matched"),
        "imdb_resolved_id",
    ].dropna()
)

principals_path = (
    IMDB_DATA_DIR
    / DATASET_FILES["title_principals"]
)

principal_columns = [
    "tconst",
    "ordering",
    "nconst",
    "category",
    "job",
    "characters",
]

relevant_principal_chunks = []

reader = pd.read_csv(
    principals_path,
    sep="\t",
    dtype="string",
    na_values="\\N",
    usecols=principal_columns,
    chunksize=CHUNK_SIZE,
    compression="gzip",
)

for chunk in tqdm(
    reader,
    desc="Scanning title.principals",
):
    relevant = chunk[
        chunk["tconst"].isin(resolved_ids)
    ].copy()

    if not relevant.empty:
        relevant_principal_chunks.append(relevant)

relevant_principals = (
    pd.concat(
        relevant_principal_chunks,
        ignore_index=True,
    )
    if relevant_principal_chunks
    else pd.DataFrame(columns=principal_columns)
)

relevant_principals["ordering"] = pd.to_numeric(
    relevant_principals["ordering"],
    errors="coerce",
).astype("Int64")

relevant_principals.to_csv(
    RELEVANT_PRINCIPALS_PATH,
    index=False,
)

print(
    f"Relevant principal-credit rows: "
    f"{len(relevant_principals):,}"
)


Scanning title.principals: 203it [02:43,  1.24it/s]


Relevant principal-credit rows: 793,506


## 11. Extract relevant director credits

In [33]:
crew_path = (
    IMDB_DATA_DIR
    / DATASET_FILES["title_crew"]
)

crew_columns = [
    "tconst",
    "directors",
    "writers",
]

relevant_crew_chunks = []

reader = pd.read_csv(
    crew_path,
    sep="\t",
    dtype="string",
    na_values="\\N",
    usecols=crew_columns,
    chunksize=CHUNK_SIZE,
    compression="gzip",
)

for chunk in tqdm(reader, desc="Scanning title.crew"):
    relevant = chunk[
        chunk["tconst"].isin(resolved_ids)
    ].copy()

    if not relevant.empty:
        relevant_crew_chunks.append(relevant)

relevant_crew = (
    pd.concat(
        relevant_crew_chunks,
        ignore_index=True,
    )
    if relevant_crew_chunks
    else pd.DataFrame(columns=crew_columns)
)

relevant_crew.to_csv(
    RELEVANT_CREW_PATH,
    index=False,
)

print(f"Relevant crew rows: {len(relevant_crew):,}")


Scanning title.crew: 26it [00:21,  1.19it/s]


Relevant crew rows: 71,357


## 12. Collect relevant person IDs

The notebook now knows which IMDb person records are needed and can scan `name.basics` without loading every person into memory.


In [34]:
principal_person_ids = set(
    relevant_principals["nconst"].dropna()
)

director_person_ids = set()

for value in relevant_crew["directors"].dropna():
    director_person_ids.update(
        item.strip()
        for item in str(value).split(",")
        if item.strip()
    )

relevant_person_ids = (
    principal_person_ids
    | director_person_ids
)

print(
    f"Relevant IMDb person IDs: "
    f"{len(relevant_person_ids):,}"
)


Relevant IMDb person IDs: 359,878


## 13. Resolve person IDs to names

In [35]:
names_path = (
    IMDB_DATA_DIR
    / DATASET_FILES["name_basics"]
)

name_columns = [
    "nconst",
    "primaryName",
    "birthYear",
    "deathYear",
    "primaryProfession",
]

relevant_name_chunks = []

reader = pd.read_csv(
    names_path,
    sep="\t",
    dtype="string",
    na_values="\\N",
    usecols=name_columns,
    chunksize=CHUNK_SIZE,
    compression="gzip",
)

for chunk in tqdm(reader, desc="Scanning name.basics"):
    relevant = chunk[
        chunk["nconst"].isin(relevant_person_ids)
    ].copy()

    if not relevant.empty:
        relevant_name_chunks.append(relevant)

relevant_names = (
    pd.concat(
        relevant_name_chunks,
        ignore_index=True,
    )
    if relevant_name_chunks
    else pd.DataFrame(columns=name_columns)
)

relevant_names.to_csv(
    RELEVANT_NAMES_PATH,
    index=False,
)

person_name_map = (
    relevant_names
    .dropna(subset=["nconst", "primaryName"])
    .drop_duplicates("nconst")
    .set_index("nconst")["primaryName"]
    .to_dict()
)

print(f"Resolved person names: {len(person_name_map):,}")


Scanning name.basics: 32it [00:54,  1.72s/it]


Resolved person names: 359,857


## 14. Build cast, director, and cinematographer fields

TV rows retain cast only. Director and cinematographer fields are intentionally blank for TV.


In [36]:
def unique_preserving_order(
    values: list[str],
) -> list[str]:
    seen = set()
    output = []

    for value in values:
        clean_value = re.sub(
            r"\s+",
            " ",
            str(value),
        ).strip()

        key = clean_value.casefold()

        if clean_value and key not in seen:
            seen.add(key)
            output.append(clean_value)

    return output


def is_cinematography_credit(
    category: Any,
    job: Any,
) -> bool:
    category_normalized = (
        str(category).strip().lower()
        if pd.notna(category)
        else ""
    )

    job_normalized = (
        str(job).strip().lower()
        if pd.notna(job)
        else ""
    )

    if category_normalized in CINEMATOGRAPHY_CATEGORIES:
        return True

    return any(
        pattern in job_normalized
        for pattern in CINEMATOGRAPHY_JOB_PATTERNS
    )


cast_by_title: dict[str, list[str]] = {}
cinematographers_by_title: dict[str, list[str]] = {}

if not relevant_principals.empty:
    ordered_principals = relevant_principals.sort_values(
        ["tconst", "ordering"],
        kind="stable",
    )

    for tconst, group in tqdm(
        ordered_principals.groupby("tconst"),
        desc="Building principal fields",
    ):
        cast_names = []
        cinematographer_names = []

        for _, credit in group.iterrows():
            person_name = person_name_map.get(
                credit["nconst"],
                "",
            )

            if not person_name:
                continue

            category = (
                str(credit["category"]).strip().lower()
                if pd.notna(credit["category"])
                else ""
            )

            if category in CAST_CATEGORIES:
                cast_names.append(person_name)

            if is_cinematography_credit(
                credit["category"],
                credit["job"],
            ):
                cinematographer_names.append(person_name)

        cast_by_title[tconst] = (
            unique_preserving_order(cast_names)[:MAX_CAST]
        )

        cinematographers_by_title[tconst] = (
            unique_preserving_order(
                cinematographer_names
            )
        )


directors_by_title: dict[str, list[str]] = {}

for _, crew_row in tqdm(
    relevant_crew.iterrows(),
    total=len(relevant_crew),
    desc="Building director fields",
):
    tconst = crew_row["tconst"]
    directors_value = crew_row["directors"]

    if pd.isna(directors_value):
        directors_by_title[tconst] = []
        continue

    director_names = [
        person_name_map.get(nconst.strip(), "")
        for nconst in str(directors_value).split(",")
        if nconst.strip()
    ]

    directors_by_title[tconst] = (
        unique_preserving_order(
            [
                name
                for name in director_names
                if name
            ]
        )
    )


Building director fields: 100%|██████████| 71357/71357 [00:07<00:00, 9343.69it/s] 


## 15. Create enriched fields for the selected rows

In [37]:
def join_people(values: list[str]) -> str:
    return " | ".join(values)


working_catalog["imdb_cast"] = (
    working_catalog["imdb_resolved_id"]
    .map(cast_by_title)
    .apply(
        lambda value: join_people(value)
        if isinstance(value, list)
        else ""
    )
)

working_catalog["imdb_directors"] = (
    working_catalog["imdb_resolved_id"]
    .map(directors_by_title)
    .apply(
        lambda value: join_people(value)
        if isinstance(value, list)
        else ""
    )
)

working_catalog["imdb_cinematographers"] = (
    working_catalog["imdb_resolved_id"]
    .map(cinematographers_by_title)
    .apply(
        lambda value: join_people(value)
        if isinstance(value, list)
        else ""
    )
)

# Requested scope:
# TV series keep cast only.
tv_mask = working_catalog["content_type"].eq("tv")

working_catalog.loc[
    tv_mask,
    [
        "imdb_directors",
        "imdb_cinematographers",
    ],
] = ""

working_catalog[
    [
        "title",
        "release_year",
        "content_type",
        "imdb_id",
        "imdb_resolved_id",
        "imdb_match_method",
        "imdb_match_status",
        "imdb_cast",
        "imdb_directors",
        "imdb_cinematographers",
    ]
].head(20)


,title,release_year,content_type,imdb_id,imdb_resolved_id,imdb_match_method,imdb_match_status,imdb_cast,imdb_directors,imdb_cinematographers
0,The Rookie,2018,tv,NaN,tt7587890,exact_title_type_year,matched,Nathan Fillion | Alyssa Diaz | Richard T. Jone...,,
1,War Machine,2026,movie,tt15940132,tt15940132,existing_imdb_id,matched,Alan Ritchson | Stephan James | Blake Richards...,Patrick Hughes,Aaron Morton
2,Grey's Anatomy,2005,tv,NaN,tt0413573,exact_title_type_year,matched,Ellen Pompeo | Chandra Wilson | James Pickens ...,,
3,Supernatural,2005,tv,NaN,tt0460681,exact_title_type_year,matched,Jared Padalecki | Jensen Ackles | Jim Beaver |...,,
4,Game of Thrones,2011,tv,NaN,tt0944947,exact_title_type_year,matched,Emilia Clarke | Peter Dinklage | Kit Harington...,,
5,The Simpsons,1989,tv,NaN,tt0096697,exact_title_type_year,matched,Dan Castellaneta | Nancy Cartwright | Julie Ka...,,
6,Bones,2005,tv,NaN,tt0460627,exact_title_type_year,matched,Emily Deschanel | David Boreanaz | Michaela Co...,,
7,Interstellar,2014,movie,tt0816692,tt0816692,existing_imdb_id,matched,Matthew McConaughey | Anne Hathaway | Jessica ...,Christopher Nolan,Hoyte van Hoytema
8,ONE PIECE,2023,tv,NaN,tt11737520,exact_title_type_year,matched,Iñaki Godoy | Emily Rudd | Mackenyu | Jacob Ro...,,
9,Inception,2010,movie,tt1375666,tt1375666,existing_imdb_id,matched,Leonardo DiCaprio | Joseph Gordon-Levitt | Ell...,Christopher Nolan,Wally Pfister


## 16. Merge enriched fields into the complete catalog

When `ROW_LIMIT` is active, only the selected rows receive results and the remaining rows are marked `not_processed`.


In [38]:
enrichment_columns = [
    "imdb_resolved_id",
    "imdb_match_method",
    "imdb_match_status",
    "imdb_match_candidates",
    "imdb_cast",
    "imdb_directors",
    "imdb_cinematographers",
]

enrichment = working_catalog[
    ["_catalog_row", *enrichment_columns]
].copy()

output_df = catalog.copy()
output_df["_catalog_row"] = output_df.index

output_df = output_df.merge(
    enrichment,
    on="_catalog_row",
    how="left",
    validate="one_to_one",
)

output_df["imdb_match_status"] = (
    output_df["imdb_match_status"]
    .fillna("not_processed")
)

for column in [
    "imdb_resolved_id",
    "imdb_match_method",
    "imdb_cast",
    "imdb_directors",
    "imdb_cinematographers",
]:
    output_df[column] = (
        output_df[column].fillna("")
    )

# Enforce movie-only crew fields once more.
output_tv_mask = output_df["content_type"].eq("tv")

output_df.loc[
    output_tv_mask,
    [
        "imdb_directors",
        "imdb_cinematographers",
    ],
] = ""

output_df = output_df.drop(columns="_catalog_row")

output_df.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved: {OUTPUT_PATH.resolve()}")
print(f"Rows: {len(output_df):,}")
print(f"Columns: {len(output_df.columns):,}")


Saved: /Users/blanca/Library/Mobile Documents/com~apple~CloudDocs/IRONHACK/github/Streaming-Marketing-Analytics/DATA/PROCESSED/all_streaming_titles_enriched.csv
Rows: 96,200
Columns: 48


## 17. Coverage summary

In [39]:
status_summary = (
    output_df
    .groupby(
        ["content_type", "imdb_match_status"],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(
        ["content_type", "rows"],
        ascending=[True, False],
    )
)

status_summary


,content_type,imdb_match_status,rows
4,movie,matched,7061
3,movie,existing_id_not_in_title_basics,355
5,movie,not_found,96
7,movie,year_mismatch,31
0,movie,ambiguous_exact_year,5
6,movie,not_found_correct_type,3
1,movie,ambiguous_nearby_year,1
2,movie,ambiguous_no_year,1
12,tv,matched,65428
13,tv,not_found,19251


In [40]:
coverage = pd.DataFrame({
    "metric": [
        "Rows resolved to an IMDb ID",
        "Movies with cast",
        "Movies with directors",
        "Movies with cinematographers",
        "TV series with cast",
    ],
    "rows": [
        output_df["imdb_resolved_id"].ne("").sum(),
        (
            output_df["content_type"].eq("movie")
            & output_df["imdb_cast"].ne("")
        ).sum(),
        (
            output_df["content_type"].eq("movie")
            & output_df["imdb_directors"].ne("")
        ).sum(),
        (
            output_df["content_type"].eq("movie")
            & output_df["imdb_cinematographers"].ne("")
        ).sum(),
        (
            output_df["content_type"].eq("tv")
            & output_df["imdb_cast"].ne("")
        ).sum(),
    ],
})

coverage


,metric,rows
0,Rows resolved to an IMDb ID,72489
1,Movies with cast,7033
2,Movies with directors,7036
3,Movies with cinematographers,6499
4,TV series with cast,62778


## 18. Review ambiguous and unmatched titles

In [41]:
review_statuses = {
    "ambiguous_exact_year",
    "ambiguous_nearby_year",
    "ambiguous_no_year",
    "year_mismatch",
    "not_found",
    "not_found_correct_type",
    "existing_id_not_in_title_basics",
}

output_df.loc[
    output_df["imdb_match_status"].isin(
        review_statuses
    ),
    [
        "title",
        "original_title",
        "release_year",
        "content_type",
        "imdb_id",
        "imdb_resolved_id",
        "imdb_match_status",
        "imdb_match_candidates",
    ],
].head(200)


,title,original_title,release_year,content_type,imdb_id,imdb_resolved_id,imdb_match_status,imdb_match_candidates
42,Obake no Q-tarō,オバケのQ太郎,1985,tv,NaN,,year_mismatch,5
45,Pili,霹靂布袋戲,1988,tv,NaN,,not_found_correct_type,0
53,Men on a Mission,아는 형님,2015,tv,NaN,,not_found,0
111,Heartland,Heartland,2007,tv,NaN,,ambiguous_exact_year,2
230,Raw,Raw,1993,tv,NaN,,year_mismatch,3
...,...,...,...,...,...,...,...,...
2226,I Can I BB,奇葩说,2014,tv,NaN,,not_found,0
2229,Electra,Ηλέκτρα,2023,tv,NaN,,not_found_correct_type,0
2235,Carl Weber's The Family Business,Carl Weber's The Family Business,2018,tv,NaN,,not_found,0
2240,The investigation was conducted,Следствие вели...,2006,tv,NaN,,not_found,0


## 19. Final quality checks

These assertions verify that:

- Every original row was retained.
- TV rows have no director or cinematographer values.
- Resolved IDs follow the IMDb title-ID format.


In [42]:
assert len(output_df) == len(catalog), (
    "The output row count differs from the input row count."
)

assert not output_df.loc[
    output_df["content_type"].eq("tv"),
    "imdb_directors",
].ne("").any(), (
    "TV rows unexpectedly contain directors."
)

assert not output_df.loc[
    output_df["content_type"].eq("tv"),
    "imdb_cinematographers",
].ne("").any(), (
    "TV rows unexpectedly contain cinematographers."
)

resolved_id_mask = output_df["imdb_resolved_id"].ne("")

assert output_df.loc[
    resolved_id_mask,
    "imdb_resolved_id",
].map(
    lambda value: bool(
        IMDB_ID_PATTERN.fullmatch(str(value))
    )
).all(), (
    "At least one resolved IMDb ID has an invalid format."
)

print("All final quality checks passed.")


All final quality checks passed.
